Label-noise robustness split generator

In [1]:
import pickle
from pathlib import Path

import numpy as np
import pandas as pd

DATA_DIR = Path("../data")
SPLITS_DIR = DATA_DIR / "splits"

SAMPLE_SIZE = 50_000
NOISE_RATE = 0.10
SAMPLE_SEED = 42
FLIP_SEED = 43

with open(DATA_DIR / "artifacts" / "label_encoder.pkl", "rb") as f:
    label_encoder = pickle.load(f)
LABEL_ORDER = list(label_encoder.classes_)
print("Labels:", LABEL_ORDER)

train_df = pd.read_csv(SPLITS_DIR / "train.csv")
print("Full train rows:", len(train_df))

Labels: ['Business', 'Entertainment', 'Health', 'Politics', 'Science', 'Sports', 'Technology', 'World']


Full train rows: 224828


In [2]:
sample_df = train_df.sample(n=SAMPLE_SIZE, random_state=SAMPLE_SEED).reset_index(drop=True)

rng = np.random.default_rng(FLIP_SEED)
n_flip = int(round(len(sample_df) * NOISE_RATE))
flip_idx = rng.choice(len(sample_df), size=n_flip, replace=False)

noisy_category = sample_df["category"].to_numpy(copy=True)
for i in flip_idx:
    true_label = noisy_category[i]
    other_labels = [c for c in LABEL_ORDER if c != true_label]
    noisy_category[i] = other_labels[rng.integers(len(other_labels))]

noisy_df = sample_df.copy()
noisy_df["category_true"] = sample_df["category"]
noisy_df["category"] = noisy_category
noisy_df["is_flipped"] = False
noisy_df.loc[flip_idx, "is_flipped"] = True

print(f"Sampled {len(noisy_df)} rows, flipped {noisy_df['is_flipped'].sum()} ({noisy_df['is_flipped'].mean():.1%})")
noisy_df["category"].value_counts()

Sampled 50000 rows, flipped 5000 (10.0%)


category
Entertainment    11955
Business          6422
Politics          6395
World             6255
Technology        6146
Sports            6063
Health            4592
Science           2172
Name: count, dtype: int64

In [3]:
out_dir = SPLITS_DIR / "noise10"
out_dir.mkdir(parents=True, exist_ok=True)

cols = ["text", "category", "category_true", "is_flipped"]
if "source_dataset" in noisy_df.columns:
    cols.append("source_dataset")

noisy_df[cols].to_csv(out_dir / "train.csv", index=False, encoding="utf-8")
print("Wrote:", (out_dir / "train.csv").resolve())
print("noise_rate:", NOISE_RATE, "| sample_size:", SAMPLE_SIZE, "| sample_seed:", SAMPLE_SEED, "| flip_seed:", FLIP_SEED)
print("Eval must use the original, unmodified data/splits/val.csv (not copied/written here).")

Wrote: D:\UoL\Sem2\mnu2\models\data\splits\noise10\train.csv
noise_rate: 0.1 | sample_size: 50000 | sample_seed: 42 | flip_seed: 43
Eval must use the original, unmodified data/splits/val.csv (not copied/written here).
